In [27]:
import json, io, base64
from PIL import Image
from shared.data.dataset import _decode_image
from shared.data.transforms import get_val_transform
from curved_road.exp1.model import Exp1Model
import torch

In [5]:
file = '/home/suresh/Projects/vizuara/vfad/2_week_simulator/curved_road/exp1/training-data/vla_dataset_128x128_1775032762942.json'

In [6]:
with open(file) as f:
    raw = json.load(f)


In [17]:
sample = raw["samples"][0]
print("actions:", sample["actions"]) 

actions: {'forward': 1, 'backward': 0, 'left': 0, 'right': 0}


In [22]:
img = _decode_image(sample["image"])                               
print("PIL image size:", img.size)     # should be (128, 128)

PIL image size: (128, 128)


In [24]:
tfm = get_val_transform()
tensor = tfm(img)
print("after trasnform: ", tensor.shape, tensor.min().item(), tensor.max().item())

after trasnform:  torch.Size([3, 128, 128]) 0.0 1.0


In [28]:
batch  = tensor.unsqueeze(0)
model = Exp1Model()
ckpt =  torch.load("../results/curved_road/exp1/best_model.pt", weights_only=True)
model.load_state_dict(ckpt)


<All keys matched successfully>

In [29]:
model.eval()
with torch.no_grad():
    logits = model(batch)
    probs = torch.sigmoid(logits)

print("logits: ", logits)                                                            
print("probs:  ", probs)   # forward (~index 0) should be near 0.9+  

logits:  tensor([[ 4.9633, -9.1165, -0.7057, -2.4525]])
probs:   tensor([[9.9306e-01, 1.0982e-04, 3.3056e-01, 7.9255e-02]])
